In [ ]:
!pip install transformers torch scikit-learn pandas

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU — Go to Runtime > Change runtime type > Select T4 GPU!")

GPU: NO GPU — Go to Runtime > Change runtime type > Select T4 GPU!


In [ ]:
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = "ncbi/MedCPT-Query-Encoder"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
encoder = AutoModel.from_pretrained(MODEL_NAME)

print("MedCPT loaded ✓")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.49k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/226k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/706k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MedCPT loaded ✓


In [ ]:
import pandas as pd
import json

BASE = "/content/drive/MyDrive/MedXChAIn"
CSVFILES = "/content/drive/MyDrive/MedXChAIn/Train&TestCSVs"

train_df = pd.read_csv(f"{CSVFILES}/train_1.csv")
test_df  = pd.read_csv(f"{CSVFILES}/test_1.csv")

with open(f"{BASE}/label_mapping.json") as f:
    label_mapping = json.load(f)  # {0: "dengue fever", ...}

NUM_LABELS = len(label_mapping)
print(f"Train: {len(train_df)} | Test: {len(test_df)} | Labels: {NUM_LABELS}")

Train: 49613 | Test: 12387 | Labels: 793


In [ ]:
import torch.nn as nn

class MedCPTClassifier(nn.Module):
    def __init__(self, encoder, num_labels):
        super().__init__()
        self.encoder = encoder
        self.classifier = nn.Sequential(
            nn.Linear(768, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, num_labels)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_embedding = outputs.last_hidden_state[:, 0, :]  # [CLS] token
        return self.classifier(cls_embedding)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MedCPTClassifier(encoder, NUM_LABELS).to(device)

print(f"Model ready — running on {device} ✓")

Model ready — running on cpu ✓


In [ ]:
import torch
import torch.nn as nn

class MedCPTClassifier(nn.Module):
    def __init__(self, encoder, num_labels):
        super().__init__()
        self.encoder = encoder
        self.classifier = nn.Sequential(
            nn.Linear(768, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, num_labels)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_embedding = outputs.last_hidden_state[:, 0, :]
        return self.classifier(cls_embedding)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MedCPTClassifier(encoder, NUM_LABELS).to(device)

print(f"Model ready — running on {device} ✓")

Model ready — running on cpu ✓


In [ ]:
from torch.utils.data import Dataset, DataLoader

class SymptomDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.texts  = df["symptoms"].fillna("").tolist()
        self.labels = df["label"].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids":      encoded["input_ids"].squeeze(),
            "attention_mask": encoded["attention_mask"].squeeze(),
            "label":          torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = SymptomDataset(train_df, tokenizer)
test_dataset  = SymptomDataset(test_df,  tokenizer)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=32)

print(f"Number of training batches: {len(train_loader)} | Number of test batches: {len(test_loader)}")

Number of training batches: 1551 | Number of test batches: 388


In [ ]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=2e-5)
criterion = nn.CrossEntropyLoss()
EPOCHS = 10

for epoch in range(EPOCHS):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for batch in train_loader:
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["label"].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct    += (outputs.argmax(1) == labels).sum().item()
        total      += labels.size(0)

    print(f"Epoch {epoch+1}/{EPOCHS} — Loss: {total_loss/len(train_loader):.4f} | Train Acc: {correct/total*100:.2f}%")

print("Training complete ✓")

In [ ]:
import os
from sklearn.metrics import classification_report

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["label"].to(device)

        outputs = model(input_ids, attention_mask)
        preds   = outputs.argmax(1)

        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

acc = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
print(f"Test Accuracy: {acc*100:.2f}%")

# Define the directory path
output_dir = f"{BASE}/LocalModels"

# Create the directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Save weights to Drive
torch.save(model.state_dict(), f"{output_dir}/hospital_3_weights.pt")
print("Weights saved → hospital_3_weights.pt ✓")

Test Accuracy: 81.23%
Weights saved → hospital_3_weights.pt ✓
